# 2 Preprocessing Data

This notebook converts the collected Google Trends keyword panel into the filtered keyword-by-month matrix used in the PCA analysis. The preprocessing applies three mechanical filters and one documented manual relevance screen:

1. remove keywords with sparse coverage,
2. remove keywords that are mostly zero,
3. remove keywords with low average search interest,
4. remove terms that are standalone firms, broad consumer/media queries, or duplicate variants that should not enter PCA separately.

Inputs are read from `data/raw/`; cleaned panels and review tables are written to `data/processed/`.


In [ ]:
# Imports
from pathlib import Path

import pandas as pd


In [ ]:
# File paths and filter settings
# The project-root lookup keeps the notebook runnable from either the repo root or notebooks/.
PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), Path.cwd().parent]
    if (candidate / "data/raw/trends_final_keywords.csv").exists()
)
RAW_DATA_DIR = PROJECT_ROOT / "data/raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data/processed"

TRENDS_PATH = RAW_DATA_DIR / "trends_final_keywords.csv"
FILTERED_TRENDS_PATH = PROCESSED_DATA_DIR / "trends_final_filtered.csv"
REVIEW_KEYWORDS_PATH = PROCESSED_DATA_DIR / "keywords_for_review.csv"
FILTERING_SUMMARY_PATH = PROCESSED_DATA_DIR / "filtering_pipeline_summary.csv"

MISSING_SHARE_THRESHOLD = 0.70
ZERO_SHARE_THRESHOLD = 0.70
MEAN_INTEREST_THRESHOLD = 15


## Load Keyword Panel

The raw file is in long format, with one monthly Google Trends interest value for each keyword/date pair. Pivoting gives the keyword-by-month matrix expected by the downstream PCA notebook.


In [ ]:
# Load keyword-level Google Trends data.
trends = pd.read_csv(TRENDS_PATH, parse_dates=["date"])

print(f"Rows: {len(trends):,}")
print(f"Keywords: {trends['keyword'].nunique():,}")
print(f"Date range: {trends['date'].min().date()} to {trends['date'].max().date()}")

trends.head()


In [ ]:
# Pivot to a keyword-by-month matrix.
trends_pivot = trends.pivot_table(
    index="keyword",
    columns="date",
    values="interest",
    aggfunc="first",
).sort_index()

initial_keyword_count = len(trends_pivot)
filtering_rows = [{
    "Stage": "Initial keyword universe",
    "Filtering criterion": "All successfully collected unique keywords",
    "Keywords before": initial_keyword_count,
    "Keywords removed": 0,
    "Keywords remaining": initial_keyword_count,
}]

print(f"Shape after pivot: {trends_pivot.shape}")
print(f"Keywords: {trends_pivot.shape[0]:,}")
print(f"Monthly observations: {trends_pivot.shape[1]:,}")

trends_pivot.head()


## Mechanical Filters

These filters remove keywords that would add noise or instability to the PCA input panel. The thresholds are intentionally defined once at the top of the notebook so the filtering choices are easy to audit.


In [ ]:
def record_filter(stage, criterion, before, after):
    # Add one row to the preprocessing audit table.
    filtering_rows.append({
        "Stage": stage,
        "Filtering criterion": criterion,
        "Keywords before": before,
        "Keywords removed": before - after,
        "Keywords remaining": after,
    })


def apply_keyword_filter(panel, drop_mask, stage, criterion):
    # Drop keywords selected by drop_mask and record the filter result.
    before = len(panel)
    filtered_panel = panel.loc[~drop_mask].copy()
    record_filter(stage, criterion, before, len(filtered_panel))
    print(f"{stage}: removed {before - len(filtered_panel):,}; remaining {len(filtered_panel):,}")
    return filtered_panel


In [ ]:
# Filter 1: remove keywords with too many missing observations.
missing_share_by_keyword = trends_pivot.isna().mean(axis=1)
trends_pivot = apply_keyword_filter(
    trends_pivot,
    missing_share_by_keyword >= MISSING_SHARE_THRESHOLD,
    "Missing-value filter",
    f"Remove keywords with at least {MISSING_SHARE_THRESHOLD:.0%} missing values",
)


In [ ]:
# Filter 2: remove keywords with too many zero observations.
zero_share_by_keyword = (trends_pivot == 0).mean(axis=1)
trends_pivot = apply_keyword_filter(
    trends_pivot,
    zero_share_by_keyword >= ZERO_SHARE_THRESHOLD,
    "Zero-value filter",
    f"Remove keywords with at least {ZERO_SHARE_THRESHOLD:.0%} zero values",
)


In [ ]:
# Inspect mean-interest cutoffs before choosing the final threshold.
mean_interest_by_keyword = trends_pivot.mean(axis=1)
thresholds = [0.5, 1, 2, 5, 10, 15, 20, 50]

mean_filter_summary = pd.DataFrame([
    {
        "mean_interest_threshold": threshold,
        "keywords_dropped": (mean_interest_by_keyword < threshold).sum(),
        "keywords_remaining": (mean_interest_by_keyword >= threshold).sum(),
    }
    for threshold in thresholds
])

print("Mean interest statistics after missing/zero filters:")
print(mean_interest_by_keyword.describe().round(2))

mean_filter_summary


In [ ]:
# Filter 3: retain keywords with sufficient average search interest.
mean_interest_by_keyword = trends_pivot.mean(axis=1)
trends_pivot = apply_keyword_filter(
    trends_pivot,
    mean_interest_by_keyword < MEAN_INTEREST_THRESHOLD,
    "Mean-interest filter",
    f"Remove keywords with mean interest below {MEAN_INTEREST_THRESHOLD}",
)

print(f"Filtered panel shape: {trends_pivot.shape}")


## Manual Relevance Screen

The mechanical filters leave some terms that are not appropriate PCA inputs for the financial-attention index. This final screen removes standalone company names, broad consumer/media terms, and duplicate variants. Company-specific search terms are retained when the query explicitly refers to the stock, such as `amazon stock` or `apple stock`.


In [ ]:
# Export the mechanically filtered keyword list for manual review.
keywords_for_review = pd.DataFrame({
    "keyword": trends_pivot.index,
    "keep": "",
})
keywords_for_review.to_csv(REVIEW_KEYWORDS_PATH, index=False)

print(f"Saved {len(keywords_for_review):,} keywords for review to {REVIEW_KEYWORDS_PATH.name}")
keywords_for_review.head(20)


In [ ]:
# Terms removed after manual relevance review.
manual_terms_to_remove = [
    "alphabet",
    "apple iphone",
    "bank of america",
    "chevron",
    "comcast",
    "credit cards",
    "disney plus",
    "disney+",
    "duke energy",
    "exxon",
    "glass",
    "google search",
    "home depot",
    "hotels",
    "meta",
    "movies",
    "netflix",
    "paper",
    "paramount plus",
    "pharmaceuticals",
    "redfin",
    "restaurants",
    "spotify",
    "stock market",
    "tiktok",
    "walmart grocery",
    "wells fargo",
    "youtube",
    "zillow",
]

removed_manual_terms = sorted(set(manual_terms_to_remove).intersection(trends_pivot.index))
trends_pivot = apply_keyword_filter(
    trends_pivot,
    trends_pivot.index.isin(manual_terms_to_remove),
    "Manual relevance screen",
    "Remove standalone firms and non-relevant consumer/media terms",
)

pd.DataFrame({"removed_keyword": removed_manual_terms})


## Save Final Dataset

The resulting matrix is the direct input for `3_pca_analysis.ipynb`: rows are keywords, columns are monthly dates, and values are normalized Google Trends interest.


In [ ]:
# Save the final filtered keyword panel.
trends_pivot.to_csv(FILTERED_TRENDS_PATH)

print(f"Final filtered dataset saved to: {FILTERED_TRENDS_PATH.name}")
print(f"Shape: {trends_pivot.shape}")
print(f"Keywords: {trends_pivot.shape[0]:,}")
print(f"Monthly observations: {trends_pivot.shape[1]:,}")
print(f"Date range: {trends_pivot.columns.min().date()} to {trends_pivot.columns.max().date()}")


In [ ]:
# Final keyword-level sanity check.
mean_interest = trends_pivot.mean(axis=1).sort_values(ascending=False)

print("Top 20 keywords by mean interest:")
print(mean_interest.head(20))

print("\nBottom 20 keywords by mean interest:")
print(mean_interest.tail(20))

print("\nMean interest statistics:")
print(mean_interest.describe().round(2))


## Filtering Pipeline Summary

The table below records each filtering step and writes the same summary to CSV for use in the README or appendix material.


In [ ]:
# Build and save the publication-ready filtering summary table.
filtering_pipeline_summary = pd.DataFrame(filtering_rows)
filtering_pipeline_summary["Share removed at stage"] = (
    filtering_pipeline_summary["Keywords removed"]
    / filtering_pipeline_summary["Keywords before"]
)
filtering_pipeline_summary["Share of initial universe retained"] = (
    filtering_pipeline_summary["Keywords remaining"] / initial_keyword_count
)

filtering_pipeline_summary.to_csv(FILTERING_SUMMARY_PATH, index=False)

print(f"Final panel: {len(trends_pivot):,} keywords x {trends_pivot.shape[1]:,} months")
print(f"Total keywords removed: {initial_keyword_count - len(trends_pivot):,}")
print(f"Filtering summary saved to: {FILTERING_SUMMARY_PATH.name}")

filtering_pipeline_summary.style.format({
    "Keywords before": "{:,.0f}",
    "Keywords removed": "{:,.0f}",
    "Keywords remaining": "{:,.0f}",
    "Share removed at stage": "{:.1%}",
    "Share of initial universe retained": "{:.1%}",
}).hide(axis="index").set_caption("Table: Google Trends Keyword Filtering Pipeline")
